<p class="lead">Deep learning has transformed parts of this laboratory's work and left other parts untouched. Structure prediction, protein language models and generative chemistry are now unthinkable without it; small tabular structure-activity datasets of a few thousand compounds are still, more often than not, best served by gradient boosting. This unit teaches the essentials from the mathematics up — the multilayer perceptron, the forward pass, backpropagation derived by hand and verified against a numerical gradient, stochastic gradient descent with Adam, and the three regularisers you will actually use — with a complete network written in <strong>base R matrices</strong> and cross-checked against <code>nnet</code>. It then runs the comparison that matters: <strong>the same scaffold split, a neural network and a gradient-boosting model, with the same preprocessing and an honestly tuned learning rate</strong>. The second half explains what a pretrained molecular or protein representation is, how frozen embeddings are used as features, and — because no pretrained weights can be downloaded in this offline environment — substitutes a transparent surrogate trained in the document, clearly labelled as such.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad presenta los fundamentos del aprendizaje profundo ejecutable en CPU, sin PyTorch. Se desarrolla el perceptrón multicapa, el paso hacia adelante y la retropropagación, con un ejemplo numérico resuelto a mano y verificado contra un gradiente numérico por diferencias finitas; luego el descenso de gradiente estocástico con Adam y la regularización mediante dropout, decaimiento de pesos y detención temprana. La red se implementa desde cero con matrices de R base y se contrasta con <code>nnet</code>. Se entrena sobre huellas moleculares de Morgan del conjunto ROCK1 con partición por andamiaje de Bemis-Murcko y se compara honestamente con un modelo de potenciación de gradiente (<code>xgboost</code>), incluyendo una búsqueda de hiperparámetros sobre el conjunto de validación, la variabilidad entre semillas y curvas de aprendizaje frente al tamaño del entrenamiento. Finalmente se explica qué son las representaciones preentrenadas (modelos de lenguaje químico tipo ChemBERTa y modelos de proteínas como ESM-2) y cómo se usan sus incrustaciones congeladas; como no es posible descargar pesos preentrenados, se utiliza un sustituto transparente (SVD y un autocodificador entrenados en la propia libreta), declarado explícitamente como sustituto.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain the multilayer perceptron, backpropagation, stochastic gradient descent, regularisation (dropout, weight decay, early stopping) and the role of the validation set.</li>
<li>Implement a small network from scratch with a correct training loop and reproducible seeds, verify its gradients, and compare it honestly with gradient boosting under the same split.</li>
<li>Explain what pretrained molecular (ChemBERTa-type) and protein (ESM-2) embeddings are, how frozen embeddings are used in a classical pipeline, and when transfer learning helps.</li>
<li>Read a deep-learning paper's methods critically: data splits, baselines, number of runs, compute and released code.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Everything here runs on a CPU in a few minutes; there is no GPU anywhere in this course. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","irlba","nnet","xgboost","glmnet","patchwork"))</code> once before the first chunk. This unit is <em>extended</em> material: skip it on a first pass if your project is a classical QSAR or a simulation analysis, and read it before you referee or write a paper that uses a neural network.</p></div>

<div class="box bridge"><h4>Substitution — no PyTorch, and why that is an opportunity</h4>
<p>The Python edition of this unit uses <strong>PyTorch</strong>: a tiny MLP, <code>loss.backward()</code>, AdamW. There is no PyTorch (and no <code>torch</code> for R, and no <code>neuralnet</code>) in this environment, so the network here is written out in <strong>base R matrix algebra</strong> — forward pass, backpropagation, dropout, Adam, early stopping, about sixty lines in total — and cross-checked two ways: its analytic gradient against a finite-difference gradient (which is what <code>loss.backward()</code> is verified against anyway), and its held-out performance against <code>nnet</code><sup class="c"><a href="#r31">[31]</a></sup>, the one neural-network package in base R's recommended set. Nothing is hidden behind a framework, which is the point of the unit; what is lost is speed and the convolutional, recurrent and attention layers a framework would give you.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(irlba); library(nnet)
  library(xgboost);    library(glmnet); library(ggplot2); library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| nnet", as.character(packageVersion("nnet")),
    "| xgboost", as.character(packageVersion("xgboost")),
    "| irlba", as.character(packageVersion("irlba")),
    "| glmnet", as.character(packageVersion("glmnet")), "\n")
cat("CPU only; threads used by xgboost:", 4, "\n")


In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)
df  <- fread(dp("rock1_chembl_curated.csv")); cmp <- df[censored == FALSE]
y <- cmp$pActivity; ycls <- as.integer(y >= 7); scaf <- cmp$scaffold_smiles
sp <- fread(dp("rock1_morgan_sparse.csv.gz")); keep <- sp[id %in% cmp$inchikey]
FP <- sparseMatrix(i = match(keep$id, cmp$inchikey), j = keep$bit + 1L, x = 1, dims = c(nrow(cmp), 2048))
Xfp <- as.matrix(FP); storage.mode(Xfp) <- "double"
scaffold_split <- function(s, fr = c(0.70, 0.15)) {   # largest Murcko families to training
  g <- split(seq_along(s), s); o <- order(-lengths(g), names(g))
  n <- length(s); n1 <- floor(fr[1]*n); n2 <- floor(fr[2]*n); tr <- va <- te <- integer(0)
  for (k in o) { i <- g[[k]]
    if (length(tr) < n1) tr <- c(tr, i) else if (length(va) < n2) va <- c(va, i) else te <- c(te, i) }
  list(tr = tr, va = va, te = te) }
S <- scaffold_split(scaf); TRs <- S$tr; VAs <- S$va; TEs <- S$te
set.seed(0); p_ <- sample(length(y))
TRr <- p_[1:floor(.70*length(y))]; VAr <- p_[(floor(.70*length(y)) + 1):floor(.85*length(y))]
TEr <- p_[(floor(.85*length(y)) + 1):length(y)]
r2  <- function(a, p) 1 - sum((a - p)^2)/sum((a - mean(a))^2)
mae <- function(a, p) mean(abs(a - p))
cat(sprintf("compounds %d, features %d bits (%.0f set per molecule on average)\n",
            length(y), ncol(Xfp), mean(rowSums(Xfp))))
cat(sprintf("scaffold split: train %d / val %d / test %d; scaffolds shared between train and test %d\n",
            length(TRs), length(VAs), length(TEs), length(intersect(scaf[TRs], scaf[TEs]))))
cat(sprintf("random split  : train %d / val %d / test %d\n", length(TRr), length(VAr), length(TEr)))
cat(sprintf("target: mean %.2f, sd %.2f; scaffold-test mean %.2f (the rare-scaffold compounds are less potent)\n",
            mean(y), sd(y), mean(y[TEs])))



## 1. Where deep learning changed the field, and where it did not

Three statements, each with evidence behind it.

**It changed structural biology.** AlphaFold2 turned protein-structure prediction from an open problem into a utility<sup class="c"><a href="#r24">[24]</a></sup>, and AlphaFold3 extended the same approach to complexes with ligands and nucleic acids<sup class="c"><a href="#r25">[25]</a></sup>. Protein language models trained on hundreds of millions of sequences predict structure directly from sequence and produce embeddings that carry functional information<sup class="c"><a href="#r22">[22]</a><a href="#r23">[23]</a></sup>.

**It changed representation learning for molecules.** Chemical language models trained on tens of millions of SMILES strings<sup class="c"><a href="#r20">[20]</a><a href="#r21">[21]</a></sup> and message-passing graph networks<sup class="c"><a href="#r17">[17]</a></sup> give molecular representations that need no hand-chosen descriptors.

**It did not change small tabular QSAR.** On datasets of the size this laboratory usually has — hundreds to a few thousand compounds, a single target, heterogeneous public measurements — gradient-boosted trees remain at least competitive, and often better, than neural networks. That is not an opinion: it is the conclusion of systematic comparisons on tabular data generally<sup class="c"><a href="#r13">[13]</a><a href="#r14">[14]</a></sup> and of the re-analysis of the largest published deep-learning-versus-classical comparison in cheminformatics, where the reported advantage shrank substantially once the validation protocol was corrected<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup>. Sections 5 to 7 reproduce that finding on our own data.

The practical consequence for a student joining the group: learn the mechanics well enough to use pretrained models and to referee papers that use them, and do not reach for a network when 2,000 rows and 2,048 sparse features are what you have.

<div class="box bridge"><h4>Domain context — what a neural network is being asked to do here</h4>
<p>Our input is a <strong>Morgan fingerprint</strong>: a 2,048-bit vector in which each bit records the presence of one circular atom environment of radius up to 2. It is sparse (about 52 bits set per molecule), binary, and its coordinates have no order or metric — bit 1,407 is not "between" bits 1,406 and 1,408. A network's first layer must therefore learn 2,048 independent weights per hidden unit from fewer than 2,000 training molecules, with no spatial or sequential structure to exploit, which is precisely the regime where the inductive biases that make deep learning powerful (locality, weight sharing, hierarchy) do not apply. The output is <strong>pActivity</strong>, the negative base-10 logarithm of the measured IC50/Ki/Kd in molar units, so 7.0 means 100 nM and an error of 1.0 is a tenfold error in potency. The measurements come from many papers and carry an experimental uncertainty of roughly half a log unit (<a href="../U17_Uncertainty_AD/index.html">Unit 17</a> measured 0.506 on this very table), which caps what any architecture can achieve.</p></div>

## 2. The multilayer perceptron

A **multilayer perceptron** (MLP) composes affine maps with a non-linear activation. For one hidden layer,

\[ \mathbf{z}^{(1)} = \mathbf{x}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}, \qquad
   \mathbf{a}^{(1)} = \phi\!\left(\mathbf{z}^{(1)}\right), \qquad
   \hat y = \mathbf{a}^{(1)}\mathbf{W}^{(2)} + b^{(2)} , \]

with \(\phi\) applied elementwise and row-vector inputs, as R's matrix algebra prefers. The activation is what makes the composition more than a single linear map: without it, stacking layers collapses to one matrix product. The standard choice is the rectified linear unit \(\phi(z)=\max(0,z)\), whose derivative is the indicator \(\mathbb{1}[z>0]\) — cheap, and free of the saturation that makes sigmoids train slowly<sup class="c"><a href="#r1">[1]</a></sup>.

For regression the loss is the mean squared error; everything else — depth, width, dropout, weight decay — is a way of controlling how a model with more parameters than data points behaves<sup class="c"><a href="#r10">[10]</a></sup>. The network below has about half a million parameters and fewer than two thousand training compounds; classical statistics says this cannot work, and sometimes it does, which is the phenomenon the field is still explaining.



In [ ]:
lay <- list(c(0.08, 6, "input\n2048 bits"), c(0.42, 4, "hidden 1\nReLU"),
            c(0.74, 3, "hidden 2\nReLU"), c(0.95, 1, "output"))
nodes <- do.call(rbind, lapply(seq_along(lay), function(i) {
  n <- as.integer(lay[[i]][2])
  data.frame(l = i, x = as.numeric(lay[[i]][1]), yy = seq(0.2, 0.8, length.out = n)) }))
edges <- do.call(rbind, lapply(1:3, function(i) {
  a <- nodes[nodes$l == i, ]; b <- nodes[nodes$l == i + 1, ]
  merge(data.frame(x = a$x, y = a$yy), data.frame(xend = b$x, yend = b$yy)) }))
g1 <- ggplot() + geom_segment(data = edges, aes(x, y, xend = xend, yend = yend),
                              colour = "grey85", linewidth = .2) +
  geom_point(data = nodes, aes(x, yy, colour = l == 1), size = 2.4, show.legend = FALSE) +
  scale_colour_manual(values = c("TRUE" = "grey55", "FALSE" = unname(PAL["blue"]))) +
  annotate("text", x = sapply(lay, function(z) as.numeric(z[1])), y = 0.06,
           label = sapply(lay, function(z) z[3]), size = 2.3) +
  ylim(0, 0.92) + labs(title = "A three-layer MLP (schematic)", x = NULL, y = NULL) +
  theme_void() + theme(plot.title = element_text(face = "bold", size = 10))
z <- seq(-3, 3, length.out = 200)
act <- rbind(data.frame(z, v = pmax(0, z), f = "ReLU"), data.frame(z, v = tanh(z), f = "tanh"),
             data.frame(z, v = 1/(1 + exp(-z)), f = "sigmoid"))
g2 <- ggplot(act, aes(z, v, colour = f)) + geom_line() + scale_colour_manual(values = unname(PAL)) +
  labs(x = "z", y = expression(phi(z)), title = "ReLU does not saturate for z > 0") +
  theme(legend.position = "bottom", legend.title = element_blank())
print(g1 | g2)
n_params <- 2048*256 + 256 + 256*64 + 64 + 64 + 1
cat(sprintf("parameters of the network used below: %s for %d training compounds (%.0f per observation)\n",
            format(n_params, big.mark = ","), length(TRs), n_params/length(TRs)))



<p class="caption"><b>Figure 1.</b> Left: the architecture used in this unit — a fully connected network from the fingerprint to a scalar prediction. Right: three activation functions; ReLU is piecewise linear with constant gradient for positive inputs, while tanh and the sigmoid flatten at both ends, which slows learning when units saturate.</p>

## 3. Backpropagation, derived and checked by hand

Training means computing \(\partial L/\partial \theta\) for every parameter. Backpropagation is the chain rule applied in reverse order, reusing intermediate results. For the two-layer network above with a squared-error loss on one example,

\[ \frac{\partial L}{\partial \hat y} = (\hat y - y), \qquad
   \frac{\partial L}{\partial \mathbf{W}^{(2)}} = \mathbf{a}^{(1)\top}(\hat y - y), \qquad
   \frac{\partial L}{\partial b^{(2)}} = (\hat y - y), \]

\[ \boldsymbol{\delta}^{(1)} = \left((\hat y - y)\,\mathbf{W}^{(2)\top}\right) \odot \phi'\!\left(\mathbf{z}^{(1)}\right), \qquad
   \frac{\partial L}{\partial \mathbf{W}^{(1)}} = \mathbf{x}^{\top}\boldsymbol{\delta}^{(1)}, \qquad
   \frac{\partial L}{\partial \mathbf{b}^{(1)}} = \boldsymbol{\delta}^{(1)} . \]

Two facts are worth internalising. The cost of the backward pass is the same order as the forward pass, which is why training a network is feasible at all<sup class="c"><a href="#r2">[2]</a></sup> — this is reverse-mode automatic differentiation, and Baydin and colleagues' survey is the general account<sup class="c"><a href="#r12">[12]</a></sup>. And the only place the loss enters is through \((\hat y - y)\): everything else is the network's own geometry.

### 3.1 Worked example by hand

Take \(\mathbf{x} = (1, 2)\), \(y = 1\), and

\[\mathbf{W}^{(1)} = \begin{pmatrix} 0.10 & -0.20 \\ 0.30 & 0.40\end{pmatrix},\quad
\mathbf{b}^{(1)} = (0.05,\, -0.05),\quad
\mathbf{W}^{(2)} = (0.50,\, -0.60)^{\top},\quad b^{(2)} = 0.10,\]

with \(L = \tfrac12(\hat y - y)^2\).

| step | computation | value |
|---|---|---|
| \(z^{(1)}_1\) | \(1(0.10) + 2(0.30) + 0.05\) | 0.75 |
| \(z^{(1)}_2\) | \(1(-0.20) + 2(0.40) - 0.05\) | 0.55 |
| \(a^{(1)}\) | \(\max(0, z)\) | (0.75, 0.55) |
| \(\hat y\) | \(0.75(0.50) + 0.55(-0.60) + 0.10\) | 0.145 |
| \(L\) | \(\tfrac12(0.145-1)^2\) | 0.3655 |
| \(\partial L/\partial \hat y\) | \(0.145 - 1\) | −0.855 |
| \(\partial L/\partial W^{(2)}\) | \(-0.855 \times (0.75, 0.55)\) | (−0.6413, −0.4703) |
| \(\delta^{(1)}\) | \(-0.855 \times (0.50, -0.60) \odot (1, 1)\) | (−0.4275, +0.5130) |
| \(\partial L/\partial W^{(1)}\) | \(\mathbf{x}^{\top}\boldsymbol{\delta}^{(1)}\) | \(\begin{pmatrix}-0.4275 & 0.5130\\ -0.8550 & 1.0260\end{pmatrix}\) |

Both ReLU units are active, so \(\phi'(z) = 1\) for both and the gradient passes through unchanged. The chunk reproduces the trace in R and then checks every one of the nine partial derivatives against a central finite difference — the test a framework's automatic differentiation has to pass too.



In [ ]:
W1 <- matrix(c(0.10, 0.30, -0.20, 0.40), 2, 2)   # rows = inputs, columns = hidden units
b1 <- c(0.05, -0.05); W2 <- matrix(c(0.50, -0.60), 2, 1); b2 <- 0.10
x <- matrix(c(1, 2), 1, 2); yt <- 1
z1 <- x %*% W1 + b1; a1 <- pmax(z1, 0); yhat <- a1 %*% W2 + b2; L <- 0.5*(yhat - yt)^2
dyhat  <- as.numeric(yhat - yt)
dW2    <- t(a1)*dyhat; db2 <- dyhat
delta1 <- (dyhat * t(W2)) * (z1 > 0)
dW1    <- t(x) %*% delta1; db1 <- delta1
cat(sprintf("forward : z1 = %s   a1 = %s   yhat = %.4f   L = %.4f\n",
            paste(round(z1, 4), collapse = " "), paste(round(a1, 4), collapse = " "), yhat, L))
cat(sprintf("backward: dL/dW2 = %s   dL/db2 = %.4f   dL/db1 = %s\n",
            paste(round(dW2, 4), collapse = " "), db2, paste(round(delta1, 4), collapse = " ")))
cat("dL/dW1 =\n"); print(round(dW1, 4))
floss <- function(par) {                        # the same loss as a function of a flat parameter vector
  W1 <- matrix(par[1:4], 2, 2); b1 <- par[5:6]; W2 <- matrix(par[7:8], 2, 1); b2 <- par[9]
  0.5*(as.numeric(pmax(x %*% W1 + b1, 0) %*% W2 + b2) - yt)^2 }
par0 <- c(as.vector(W1), b1, as.vector(W2), b2); eps <- 1e-6
num <- sapply(seq_along(par0), function(i) {
  pp <- par0; pp[i] <- pp[i] + eps; pm <- par0; pm[i] <- pm[i] - eps
  (floss(pp) - floss(pm))/(2*eps) })
ana <- c(as.vector(dW1), as.vector(db1), as.vector(dW2), db2)
cat(sprintf("\nanalytic vs central-difference gradient: max |difference| = %.2e over %d parameters\n",
            max(abs(ana - num)), length(par0)))



The hand calculation, the R code and the numerical gradient agree to eleven decimal places. That is the verification worth doing once in your life: after it, `backward()` in any framework is not magic, it is the table above executed in reverse — and a finite-difference check is how you debug a custom layer when it is not.

## 4. A network in base R: the training loop

Four pieces turn the gradient into a model. **Mini-batch stochastic gradient descent**<sup class="c"><a href="#r3">[3]</a><a href="#r4">[4]</a></sup> trades gradient accuracy for many more steps per unit of computation. **Adam**<sup class="c"><a href="#r5">[5]</a></sup> adapts the step size per parameter from running estimates of the gradient's first and second moments; decoupled weight decay<sup class="c"><a href="#r6">[6]</a></sup> adds the penalty to the gradient before the moment update. **He initialisation**<sup class="c"><a href="#r8">[8]</a></sup> draws weights with variance \(2/n_{\text{in}}\) so activations neither vanish nor explode across layers. **Early stopping on a validation set, with restoration of the best weights**<sup class="c"><a href="#r7">[7]</a></sup> is the cheapest regulariser and the one most often implemented incorrectly.



In [ ]:
mlp_init <- function(n_in, h1, h2, seed = 1) { set.seed(seed)      # He initialisation
  list(W1 = matrix(rnorm(n_in*h1, 0, sqrt(2/n_in)), n_in, h1), b1 = rep(0, h1),
       W2 = matrix(rnorm(h1*h2, 0, sqrt(2/h1)), h1, h2),       b2 = rep(0, h2),
       W3 = matrix(rnorm(h2, 0, sqrt(2/h2)), h2, 1),           b3 = 0) }
mlp_forward <- function(P, X, drop = 0, train = FALSE) {
  z1 <- sweep(X %*% P$W1, 2, P$b1, "+"); a1 <- pmax(z1, 0)
  m1 <- if (train && drop > 0) (matrix(runif(length(a1)), nrow(a1)) > drop)/(1 - drop) else NULL
  if (!is.null(m1)) a1 <- a1*m1                                     # inverted dropout
  z2 <- sweep(a1 %*% P$W2, 2, P$b2, "+"); a2 <- pmax(z2, 0)
  m2 <- if (train && drop > 0) (matrix(runif(length(a2)), nrow(a2)) > drop)/(1 - drop) else NULL
  if (!is.null(m2)) a2 <- a2*m2
  list(z1 = z1, a1 = a1, z2 = z2, a2 = a2, m1 = m1, m2 = m2, out = as.numeric(a2 %*% P$W3 + P$b3)) }
cat("forward pass defined: two ReLU layers with inverted dropout, a linear output\n")


In [ ]:
mlp_backward <- function(P, X, yy, F_) {
  n <- nrow(X); d <- (F_$out - yy)/n                                # dL/d(out) for the mean squared error
  gW3 <- t(F_$a2) %*% d; gb3 <- sum(d)
  da2 <- outer(d, as.numeric(P$W3)); if (!is.null(F_$m2)) da2 <- da2*F_$m2
  dz2 <- da2*(F_$z2 > 0); gW2 <- t(F_$a1) %*% dz2; gb2 <- colSums(dz2)
  da1 <- dz2 %*% t(P$W2);  if (!is.null(F_$m1)) da1 <- da1*F_$m1
  dz1 <- da1*(F_$z1 > 0); gW1 <- t(X) %*% dz1; gb1 <- colSums(dz1)
  list(W1 = gW1, b1 = gb1, W2 = gW2, b2 = gb2, W3 = gW3, b3 = gb3) }
cat("forward and backward defined; the backward pass is the chain rule of Section 3, vectorised\n")


In [ ]:
mlp_train <- function(X, tr, va, h1 = 256, h2 = 64, drop = 0.3, lr = 1e-3, wd = 1e-4,
                      epochs = 150, patience = 25, batch = 64, seed = 1) {
  set.seed(seed); P <- mlp_init(ncol(X), h1, h2, seed)
  M <- lapply(P, function(z) z*0); V <- M; t_ <- 0
  best <- Inf; bestP <- P; bad <- 0; hist <- list()
  for (ep in seq_len(epochs)) {
    perm <- sample(length(tr))
    for (i in seq(1, length(tr), by = batch)) {
      idx <- tr[perm[i:min(i + batch - 1, length(tr))]]
      F_ <- mlp_forward(P, X[idx, , drop = FALSE], drop, TRUE)
      G  <- mlp_backward(P, X[idx, , drop = FALSE], y[idx], F_)
      t_ <- t_ + 1
      for (nm in names(P)) {                                        # AdamW update
        g <- G[[nm]] + wd*P[[nm]]
        M[[nm]] <- 0.9*M[[nm]] + 0.1*g; V[[nm]] <- 0.999*V[[nm]] + 0.001*g^2
        P[[nm]] <- P[[nm]] - lr*(M[[nm]]/(1 - 0.9^t_))/(sqrt(V[[nm]]/(1 - 0.999^t_)) + 1e-8) } }
    trl <- mean((mlp_forward(P, X[tr, , drop = FALSE])$out - y[tr])^2)   # eval mode: no dropout
    val <- mean((mlp_forward(P, X[va, , drop = FALSE])$out - y[va])^2)
    hist[[ep]] <- c(ep, trl, val)
    if (val < best - 1e-4) { best <- val; bestP <- P; bad <- 0
    } else { bad <- bad + 1; if (bad >= patience) break } }
  list(P = bestP, hist = do.call(rbind, hist), best = best) }           # best weights restored
mlp_pred <- function(P, X, idx) mlp_forward(P, X[idx, , drop = FALSE])$out
t0 <- proc.time()[["elapsed"]]
m_raw <- mlp_train(Xfp, TRs, VAs, drop = 0.3, lr = 1e-3)
pr_raw <- mlp_pred(m_raw$P, Xfp, TEs)
cat(sprintf("MLP on raw fingerprint bits, scaffold split: %d epochs in %.1f s\n",
            nrow(m_raw$hist), proc.time()[["elapsed"]] - t0))
cat(sprintf("  best validation MSE %.3f at epoch %d\n", m_raw$best, which.min(m_raw$hist[, 3])))
cat(sprintf("  test R2 %+.3f   MAE %.3f   RMSE %.3f log units\n",
            r2(y[TEs], pr_raw), mae(y[TEs], pr_raw), sqrt(mean((y[TEs] - pr_raw)^2))))



A correct loop has five elements, all visible above: a seed, mini-batch shuffling, a fresh gradient each batch (there is no accumulated state to zero because the gradient is recomputed, which is one error the R version cannot make), evaluation with dropout switched off, and early stopping that **restores** the best weights rather than returning the last ones.

The first honest result of the unit: a reasonable MLP on raw fingerprint bits, with early stopping, **fails on the scaffold split** — its \(R^2\) is negative, meaning it predicts worse than the training mean on compounds from unseen scaffold families. Before concluding anything about deep learning, three standard causes have to be excluded: insufficient regularisation, a badly chosen learning rate, and an unsuitable input representation. Sections 5 and 6 test all three.

## 5. Regularisation: dropout, weight decay, early stopping

Three mechanisms, three different jobs. **Weight decay** shrinks the weights at every step, preferring smoother functions. **Dropout** sets each hidden unit to zero with probability \(p\) during training only, rescaling the rest, so no unit can rely on the presence of another<sup class="c"><a href="#r9">[9]</a></sup>; at evaluation the full network is used, which is what the `train = FALSE` branch of `mlp_forward` does. **Early stopping** uses the validation loss as a clock<sup class="c"><a href="#r7">[7]</a></sup>.



In [ ]:
cfg <- list("none (dropout 0, wd 0)" = c(0, 0), "weight decay 1e-3" = c(0, 1e-3),
            "dropout 0.5" = c(0.5, 0), "dropout 0.5 + wd 1e-4" = c(0.5, 1e-4))
reg <- list(); regh <- list()
for (nm in names(cfg)) {
  m <- mlp_train(Xfp, TRs, VAs, drop = cfg[[nm]][1], wd = cfg[[nm]][2], epochs = 45, patience = 45)
  regh[[nm]] <- m$hist; pr <- mlp_pred(m$P, Xfp, TEs)
  reg[[length(reg) + 1]] <- data.frame(regularisation = nm, epochs = nrow(m$hist),
    final_train_MSE = m$hist[nrow(m$hist), 2], best_val_MSE = m$best,
    final_val_MSE = m$hist[nrow(m$hist), 3], test_R2 = r2(y[TEs], pr)) }
reg <- do.call(rbind, reg)
knitr::kable(reg, digits = 3, row.names = FALSE, caption = "Four regularisation settings, same architecture")
hh <- do.call(rbind, lapply(names(regh), function(nm)
  data.frame(cfg = nm, epoch = regh[[nm]][, 1], train = regh[[nm]][, 2], val = regh[[nm]][, 3])))
g1 <- ggplot(hh, aes(epoch, train, colour = cfg)) + geom_line(linewidth = .5) +
  scale_colour_manual(values = unname(PAL)) + labs(y = "training MSE", title = "Training loss") +
  theme(legend.position = "none")
best_pts <- do.call(rbind, lapply(names(regh), function(nm) { h <- regh[[nm]]
  data.frame(cfg = nm, epoch = h[which.min(h[, 3]), 1], val = min(h[, 3])) }))
g2 <- ggplot(hh, aes(epoch, val, colour = cfg)) + geom_line(linewidth = .5) +
  geom_point(data = best_pts, size = 1.8) + scale_colour_manual(values = unname(PAL)) +
  labs(y = "validation MSE", title = "Validation loss (dots = early-stopping point)") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6))
print(g1 | g2)
cat("gap between final training and validation MSE: ",
    paste(sprintf("%s %+.2f", reg$regularisation, reg$final_val_MSE - reg$final_train_MSE),
          collapse = "; "), "\n")



<p class="caption"><b>Figure 2.</b> Left: training MSE per epoch. Without regularisation it falls towards zero — the network memorises the training compounds exactly. Right: validation MSE, with the early-stopping point of each run marked; the unregularised run reaches its best validation loss within the first few epochs and then deteriorates, while the dropout runs train more slowly and plateau. The printed train–validation gaps quantify the memorisation.</p>

The unregularised curve is the canonical overfitting picture and the reason early stopping exists: the training MSE falls below 0.1 while the validation MSE sits around 1.0, a gap of about one log unit squared. Dropout and weight decay close about a third of that gap and move the held-out R² from −0.40 to roughly −0.03 — better, and still worse than predicting the training mean. That is the point: regularisation controls memorisation, it cannot create transfer to chemistry the model has never seen. The next section changes what the network sees rather than how it is penalised.

### 5.1 Two settings that break training silently

Before blaming the architecture, check the learning rate and the initialisation. Too large a step makes the validation loss oscillate; too small a one leaves the model where it started; an all-zeros initialisation leaves every hidden unit identical for ever, because symmetric units receive identical gradients.



In [ ]:
lrh <- list()
for (lr in c(1e-5, 1e-3, 1e-1)) {
  m <- mlp_train(Xfp, TRs, VAs, drop = 0.5, lr = lr, epochs = 40, patience = 40)
  lrh[[format(lr)]] <- m$hist
  cat(sprintf("lr = %.0e: best validation MSE %.3f at epoch %2d, final training MSE %.3f\n",
              lr, m$best, which.min(m$hist[, 3]), m$hist[nrow(m$hist), 2])) }
d1 <- do.call(rbind, lapply(names(lrh), function(k)
  data.frame(what = paste("lr =", k), epoch = lrh[[k]][, 1], val = lrh[[k]][, 3])))
g1 <- ggplot(d1, aes(epoch, val, colour = what)) + geom_line() + scale_y_log10() +
  scale_colour_manual(values = unname(PAL)) + labs(y = "validation MSE", title = "Learning rate") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))


In [ ]:
inith <- list()
for (lab in c("default (He)", "all zeros", "inflated (x20)")) {
  set.seed(1); P <- mlp_init(ncol(Xfp), 256, 64, 1)
  if (lab == "all zeros") P <- lapply(P, function(z) z*0)
  if (startsWith(lab, "inflated")) P[c("W1","W2","W3")] <- lapply(P[c("W1","W2","W3")], function(z) z*20)
  M <- lapply(P, function(z) z*0); V <- M; t_ <- 0; h <- list()
  for (ep in 1:25) {
    set.seed(100 + ep); perm <- sample(length(TRs))
    for (i in seq(1, length(TRs), by = 64)) {
      idx <- TRs[perm[i:min(i + 63, length(TRs))]]
      F_ <- mlp_forward(P, Xfp[idx, , drop = FALSE]); G <- mlp_backward(P, Xfp[idx, , drop = FALSE], y[idx], F_)
      t_ <- t_ + 1
      for (nm in names(P)) { g <- G[[nm]]
        M[[nm]] <- 0.9*M[[nm]] + 0.1*g; V[[nm]] <- 0.999*V[[nm]] + 0.001*g^2
        P[[nm]] <- P[[nm]] - 1e-3*(M[[nm]]/(1 - 0.9^t_))/(sqrt(V[[nm]]/(1 - 0.999^t_)) + 1e-8) } }
    h[[ep]] <- c(ep, mean((mlp_forward(P, Xfp[VAs, ])$out - y[VAs])^2)) }
  inith[[lab]] <- do.call(rbind, h)
  cat(sprintf("initialisation %-16s validation MSE after 25 epochs %.3f\n", lab, inith[[lab]][25, 2])) }
d2 <- do.call(rbind, lapply(names(inith), function(k)
  data.frame(what = k, epoch = inith[[k]][, 1], val = inith[[k]][, 2])))
g2 <- ggplot(d2, aes(epoch, val, colour = what)) + geom_line() + scale_y_log10() +
  scale_colour_manual(values = unname(PAL)) + labs(y = "validation MSE", title = "Initialisation") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
print(g1 | g2)



<p class="caption"><b>Figure 3.</b> Validation MSE on a logarithmic scale. Left: three learning rates spanning four orders of magnitude with everything else fixed — the largest is unstable, the smallest has barely moved within the epoch budget, and only the middle one behaves. Right: three initialisations; the all-zeros network stays near the loss of predicting zero (32 against 1.1 for the default) because every hidden unit receives an identical gradient and only the output bias can move, and the inflated initialisation sits five orders of magnitude worse and must first be undone by the optimiser. Neither failure produces an error message, only a bad number.</p>

## 6. Representation beats depth: the comparison with gradient boosting

Two changes, neither of them architectural. First, **reduce and scale the input**: project the 2,048 binary bits onto their 256 leading singular vectors with `irlba` (fitted on the training set only — doing it on all the data is leakage) and standardise the result, so the first layer sees 256 informative coordinates instead of 2,048 sparse indicators. Second, **select the learning rate and the dropout on the validation set**, not by instinct.

Then the comparison the reader deserves: the tuned network against `xgboost`<sup class="c"><a href="#r32">[32]</a></sup> with default-ish settings, on both splits, with the same training compounds — plus `nnet`<sup class="c"><a href="#r31">[31]</a></sup>, itself tuned on the validation set, as an independent implementation of the same idea.



In [ ]:
svd_features <- function(tr, nc = 256) {
  set.seed(0); sv <- irlba(FP[tr, ], nv = nc); Z <- as.matrix(FP %*% sv$v)
  sweep(sweep(Z, 2, colMeans(Z[tr, ])), 2, apply(Z[tr, ], 2, sd) + 1e-6, "/") }
Zs <- svd_features(TRs)
sweep_tab <- do.call(rbind, lapply(c(0.2, 0.5), function(dr) do.call(rbind, lapply(c(3e-4, 1e-3), function(lr) {
  m <- mlp_train(Zs, TRs, VAs, drop = dr, lr = lr, epochs = 150, patience = 25)
  data.frame(dropout = dr, lr = lr, epochs = nrow(m$hist), val_MSE = m$best,
             test_R2 = r2(y[TEs], mlp_pred(m$P, Zs, TEs))) }))))
sweep_tab <- sweep_tab[order(sweep_tab$val_MSE), ]
knitr::kable(sweep_tab, digits = 4, row.names = FALSE,
             caption = "Validation-set sweep on the SVD features (scaffold split)")
best_cfg <- sweep_tab[1, ]
cat(sprintf("selected on validation MSE: dropout %.1f, lr %.0e\n", best_cfg$dropout, best_cfg$lr))
nnet_tab <- do.call(rbind, lapply(c(8, 16), function(sz) do.call(rbind, lapply(c(0.1, 1), function(dc) {
  set.seed(1); nn <- nnet(x = Zs[TRs, ], y = y[TRs], size = sz, linout = TRUE, decay = dc,
                          maxit = 200, MaxNWts = 20000, trace = FALSE)
  data.frame(size = sz, decay = dc, val_R2 = r2(y[VAs], as.numeric(predict(nn, Zs[VAs, ]))),
             test_R2 = r2(y[TEs], as.numeric(predict(nn, Zs[TEs, ])))) }))))
knitr::kable(nnet_tab[order(-nnet_tab$val_R2), ], digits = 3, row.names = FALSE,
             caption = "nnet on the same features, tuned on the validation set")


In [ ]:
rows <- list()
for (spl in c("scaffold", "random")) {
  idx <- if (spl == "scaffold") list(TRs, VAs, TEs) else list(TRr, VAr, TEr)
  tr <- idx[[1]]; va <- idx[[2]]; te <- idx[[3]]
  Z <- svd_features(tr)
  nb <- mlp_train(Xfp, tr, va, drop = 0.5, lr = 1e-3, epochs = 100, patience = 20)
  nz <- mlp_train(Z,   tr, va, drop = best_cfg$dropout, lr = best_cfg$lr, epochs = 150, patience = 25)
  gb <- xgb.train(list(eta = 0.05, max_depth = 6, subsample = 0.9, colsample_bytree = 0.5,
                       objective = "reg:squarederror", nthread = 4),
                  xgb.DMatrix(Xfp[tr, ], label = y[tr]), nrounds = 300, verbose = 0)
  for (z in list(list("MLP on raw 2048 bits", mlp_pred(nb$P, Xfp, te)),
                 list("MLP on 256 scaled SVD components", mlp_pred(nz$P, Z, te)),
                 list("xgboost on raw 2048 bits", predict(gb, xgb.DMatrix(Xfp[te, ]))),
                 list("predict the training mean", rep(mean(y[tr]), length(te)))))
    rows[[length(rows) + 1]] <- data.frame(split = spl, model = z[[1]], test_R2 = r2(y[te], z[[2]]),
                                           MAE = mae(y[te], z[[2]]),
                                           RMSE = sqrt(mean((y[te] - z[[2]])^2)))
  if (spl == "scaffold") { p_mlp <<- mlp_pred(nz$P, Z, te); p_gbm <<- predict(gb, xgb.DMatrix(Xfp[te, ])) } }
comp <- do.call(rbind, rows)
knitr::kable(comp, digits = 3, row.names = FALSE, caption = "Four models, two splits, same compounds")
sc <- comp[comp$split == "scaffold", ]
g1 <- ggplot(comp, aes(reorder(model, test_R2), test_R2, fill = split)) +
  geom_col(position = position_dodge(width = .8), width = .7) + coord_flip() +
  geom_hline(yintercept = 0, linewidth = .3) + scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = NULL, y = "test R^2", title = "Same features, same compounds, two splits") +
  theme(axis.text.y = element_text(size = 6.5), legend.position = "bottom", legend.title = element_blank())
pp <- data.frame(obs = y[TEs], mlp = p_mlp, gbm = p_gbm)
g2 <- ggplot(pp) +
  geom_point(aes(obs, mlp, colour = sprintf("MLP (R^2 %+.2f)", r2(y[TEs], p_mlp))), size = .8, alpha = .6) +
  geom_point(aes(obs, gbm, colour = sprintf("xgboost (R^2 %+.2f)", r2(y[TEs], p_gbm))), size = .8, alpha = .6) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) +
  scale_colour_manual(values = unname(PAL[c("green", "blue")])) +
  labs(x = "measured pActivity", y = "predicted pActivity", title = "Scaffold test set, parity") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
print(g1 | g2)
cat(sprintf("correlation between the two models' predictions: %.3f; their average: R2 %+.3f\n",
            cor(p_mlp, p_gbm), r2(y[TEs], (p_mlp + p_gbm)/2)))



<p class="caption"><b>Figure 4.</b> Left: test R² of four models on both splits, including the trivial baseline of predicting the training mean. Right: parity plot on the scaffold test set for the tuned MLP and for xgboost; both regress towards the mean on the least potent compounds, which is what a scaffold split does to any similarity-based model. The printed line gives the correlation between the two models' predictions and the R² of their simple average.</p>

Three conclusions, in the order a referee would want them.

1. **The representation, not the architecture, was the problem.** Projecting the sparse bits onto 256 scaled components turned a network that predicted worse than the mean into one with a positive held-out R², without changing a single layer. On sparse binary inputs this preprocessing matters more than depth, width or dropout — and `nnet`, a completely independent implementation tuned the same way, lands in the same region on the same features (its best validation configuration reaches a test R² near zero), which is the cross-check the from-scratch code needed.
2. **Gradient boosting still wins, on both splits.** The margin is clear, and it is obtained with no tuning and in a fraction of the time. This is the finding of the tabular-data literature<sup class="c"><a href="#r13">[13]</a><a href="#r14">[14]</a></sup> and of the corrected cheminformatics comparison<sup class="c"><a href="#r16">[16]</a></sup>, reproduced here.
3. **The split dominates everything.** Every model loses a large share of its R² moving from the random to the scaffold split, and the MLP on raw bits collapses below zero. The difference between architectures is smaller than the difference between validation protocols, and a paper that reports a deep model's advantage under a random split has not yet made a claim about new chemistry<sup class="c"><a href="#r30">[30]</a></sup>.

### 6.1 How large is a difference of 0.03 in R²?

Retraining the same network with a different seed changes its test score, and the size of that change is the yardstick for any claimed improvement.



In [ ]:
seed_rows <- list()
for (s in 1:3) {
  m <- mlp_train(Zs, TRs, VAs, drop = best_cfg$dropout, lr = best_cfg$lr, epochs = 150,
                 patience = 25, seed = s)
  seed_rows[[length(seed_rows) + 1]] <- data.frame(seed = s, model = "MLP (SVD)",
                                                   test_R2 = r2(y[TEs], mlp_pred(m$P, Zs, TEs))) }
for (s in 1:3) {
  gb <- xgb.train(list(eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.5,
                       objective = "reg:squarederror", nthread = 4, seed = s),
                  xgb.DMatrix(Xfp[TRs, ], label = y[TRs]), nrounds = 300, verbose = 0)
  seed_rows[[length(seed_rows) + 1]] <- data.frame(seed = s, model = "xgboost",
    test_R2 = r2(y[TEs], predict(gb, xgb.DMatrix(Xfp[TEs, ])))) }
seeds <- do.call(rbind, seed_rows)
agg <- do.call(data.frame, aggregate(test_R2 ~ model, seeds,
                                     function(v) c(mean = mean(v), sd = sd(v), min = min(v), max = max(v))))
names(agg) <- c("model", "mean", "sd", "min", "max")
knitr::kable(agg, digits = 4, row.names = FALSE, caption = "Three seeds each on the scaffold split")
print(ggplot(seeds, aes(model, test_R2, colour = model)) +
        geom_point(position = position_jitter(width = .06), size = 2, show.legend = FALSE) +
        stat_summary(fun = mean, geom = "crossbar", width = .4, linewidth = .3, show.legend = FALSE) +
        scale_colour_manual(values = unname(PAL[c("green", "blue")])) +
        labs(x = NULL, y = "test R^2 (scaffold split)",
             title = "Three seeds each: the spread is the unit of comparison"))
cat(sprintf("spread of the MLP across seeds: %.3f R2 units; difference between the two model means: %+.3f\n",
            agg$max[agg$model == "MLP (SVD)"] - agg$min[agg$model == "MLP (SVD)"],
            agg$mean[agg$model == "xgboost"] - agg$mean[agg$model == "MLP (SVD)"]))



<p class="caption"><b>Figure 5.</b> Test R² of three independently seeded runs of the tuned MLP and of three differently seeded xgboost models (with row and column subsampling on, so that the seed matters), with the means marked. The printed spread across seeds is the yardstick against which any reported architectural improvement must be measured.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Keep the architecture and change only the number of SVD components (64, 128, 256, 512) in <code>svd_features</code>. Plot validation MSE against the number of components and report where it bottoms out. Then remove the standardisation (return the raw <code>Z</code> instead of the scaled one) and report how much of the gain in Section 6 came from the projection and how much from the scaling — on these inputs the second is usually larger than students expect.</p></div>

## 7. Learning curves: how much data would it take?

The question behind "should we use deep learning" is usually "would more data change the answer". A learning curve — performance against training-set size, with the same validation and test sets — is the way to ask it.



In [ ]:
sizes <- c(200, 500, 1000, length(TRs))
set.seed(0)
lc <- do.call(rbind, lapply(sizes, function(n) {
  sub <- if (n < length(TRs)) sample(TRs, n) else TRs
  Zn <- svd_features(sub, nc = min(128, n %/% 2))      # the code cannot be wider than the data
  mn <- mlp_train(Zn, sub, VAs, drop = best_cfg$dropout, lr = best_cfg$lr, epochs = 100, patience = 20)
  gn <- xgb.train(list(eta = 0.05, max_depth = 6, subsample = 0.9, colsample_bytree = 0.5,
                       objective = "reg:squarederror", nthread = 4),
                  xgb.DMatrix(Xfp[sub, ], label = y[sub]), nrounds = 300, verbose = 0)
  rn <- cv.glmnet(Xfp[sub, ], y[sub], alpha = 0, nfolds = 3)
  data.frame(n_train = n,
             MLP_SVD = r2(y[TEs], mlp_pred(mn$P, Zn, TEs)),
             xgboost = r2(y[TEs], predict(gn, xgb.DMatrix(Xfp[TEs, ]))),
             ridge_on_bits = r2(y[TEs], as.numeric(predict(rn, newx = Xfp[TEs, ], s = "lambda.min")))) }))
knitr::kable(lc, digits = 3, row.names = FALSE, caption = "Learning curves on the scaffold split")
ll <- do.call(rbind, lapply(c("MLP_SVD", "xgboost", "ridge_on_bits"),
                            function(k) data.frame(model = k, n = lc$n_train, r2 = lc[[k]])))
print(ggplot(ll, aes(n, r2, colour = model, shape = model)) + geom_line() + geom_point() +
        geom_hline(yintercept = 0, linewidth = .3) + scale_colour_manual(values = unname(PAL)) +
        labs(x = "training compounds", y = "test R^2 (scaffold split)",
             title = "Learning curves: more data helps everyone, and nobody enough") +
        theme(legend.position = "bottom", legend.title = element_blank()))
cat("R2 gained from the smallest to the largest training set: ",
    paste(sprintf("%s %+.3f", c("MLP", "xgboost", "ridge"),
                  c(lc$MLP_SVD[4] - lc$MLP_SVD[1], lc$xgboost[4] - lc$xgboost[1],
                    lc$ridge_on_bits[4] - lc$ridge_on_bits[1])), collapse = "; "), "\n")



<p class="caption"><b>Figure 6.</b> Test R² on the scaffold test set against the number of training compounds, for the MLP on SVD features, xgboost and a ridge baseline on raw bits (the SVD width is capped at half the sample size, so the code cannot be wider than the data). None of the curves has flattened at the full training-set size; ridge is ahead of xgboost at 200 compounds and behind it from 500 onwards, and the network is far below both everywhere, catastrophically so in the small-sample regime.</p>

Three readings. First, the curves not having flattened means the dataset is the binding constraint, not the model class: the right investment is more measurements (or data from related kinases), not a deeper network. Second, the small-sample behaviour of the network is the warning — at 200 training compounds its R² is about −2.6 against −0.09 for xgboost and −0.03 for ridge, and that is the regime most thesis projects actually occupy. Third, this is exactly the regime in which a *pretrained* representation is supposed to help, which is the subject of the next section.

## 8. Pretrained representations: what they are, and an offline stand-in

The most useful thing deep learning currently offers a small laboratory is not a model to train but a **representation to borrow**. The pattern is always the same:

1. A large model is pretrained on a *self-supervised* task over a huge unlabelled corpus — masked-token prediction over tens of millions of SMILES strings for chemical language models such as ChemBERTa<sup class="c"><a href="#r20">[20]</a></sup> and MoLFormer<sup class="c"><a href="#r21">[21]</a></sup>, or over hundreds of millions of protein sequences for ESM-2<sup class="c"><a href="#r22">[22]</a><a href="#r23">[23]</a></sup>.
2. Its internal activations for a new molecule or sequence — the **embedding**, typically a few hundred to a few thousand numbers — are extracted once, with the weights **frozen**.
3. Those embeddings become the feature matrix of an ordinary model: ridge regression, random forest, gradient boosting. No gradients, no GPU, no fine-tuning.

That third step is what makes the approach practical here: embedding 3,000 molecules is a single forward pass, after which the modelling is an ordinary supervised workflow. Transfer learning of this kind helps most when the labelled set is small and the pretraining corpus covers the relevant chemistry or protein family; it helps least when the labelled set is already large and homogeneous, which is the regime of most single-target QSAR<sup class="c"><a href="#r27">[27]</a><a href="#r28">[28]</a></sup>.

<div class="box bridge"><h4>Substitution, stated explicitly</h4>
<p><strong>This unit does not use a pretrained model.</strong> The course's build environment has no network access, so neither ChemBERTa nor ESM-2 weights can be downloaded, and nothing of that kind is bundled with the frozen datasets. What follows is therefore a <em>transparent surrogate</em>: a truncated-SVD embedding and an unsupervised <strong>autoencoder</strong><sup class="c"><a href="#r26">[26]</a></sup>, both trained in this document on the <em>training</em> fingerprints only. They reproduce the <em>mechanics</em> of the frozen-embedding workflow — unsupervised objective, frozen encoder, classical model on top — and nothing else. They are not pretrained models: they see about two thousand molecules, not tens of millions, and no chemistry beyond this dataset. Do not read the numbers below as evidence about ChemBERTa or ESM-2; read them as evidence about what compression alone can and cannot buy.</p></div>



In [ ]:
ae_init <- function(d_in, h, k, seed = 1) { set.seed(seed)
  list(W1 = matrix(rnorm(d_in*h, 0, sqrt(2/d_in)), d_in, h), b1 = rep(0, h),
       W2 = matrix(rnorm(h*k, 0, sqrt(2/h)), h, k),          b2 = rep(0, k),
       W3 = matrix(rnorm(k*h, 0, sqrt(2/k)), k, h),          b3 = rep(0, h),
       W4 = matrix(rnorm(h*d_in, 0, sqrt(2/h)), h, d_in),    b4 = rep(0, d_in)) }
ae_run <- function(P, X) { h1 <- pmax(sweep(X %*% P$W1, 2, P$b1, "+"), 0)
  code <- sweep(h1 %*% P$W2, 2, P$b2, "+")
  h2 <- pmax(sweep(code %*% P$W3, 2, P$b3, "+"), 0)
  list(h1 = h1, code = code, h2 = h2, out = sweep(h2 %*% P$W4, 2, P$b4, "+")) }
ae_train <- function(X, tr, va, h = 128, k = 32, epochs = 40, lr = 1e-3, batch = 128, seed = 1) {
  set.seed(seed); P <- ae_init(ncol(X), h, k, seed); M <- lapply(P, function(z) z*0); V <- M; t_ <- 0
  hist <- list()
  for (ep in seq_len(epochs)) {
    perm <- sample(length(tr))
    for (i in seq(1, length(tr), by = batch)) {
      idx <- tr[perm[i:min(i + batch - 1, length(tr))]]; B <- X[idx, , drop = FALSE]; n <- nrow(B)
      F_ <- ae_run(P, B); d <- 2*(F_$out - B)/n                       # squared reconstruction loss
      gW4 <- t(F_$h2) %*% d; gb4 <- colSums(d)
      dh2 <- (d %*% t(P$W4))*(F_$h2 > 0); gW3 <- t(F_$code) %*% dh2; gb3 <- colSums(dh2)
      dc  <- dh2 %*% t(P$W3); gW2 <- t(F_$h1) %*% dc; gb2 <- colSums(dc)
      dh1 <- (dc %*% t(P$W2))*(F_$h1 > 0); gW1 <- t(B) %*% dh1; gb1 <- colSums(dh1)
      G <- list(W1 = gW1, b1 = gb1, W2 = gW2, b2 = gb2, W3 = gW3, b3 = gb3, W4 = gW4, b4 = gb4)
      t_ <- t_ + 1
      for (nm in names(P)) { M[[nm]] <- 0.9*M[[nm]] + 0.1*G[[nm]]; V[[nm]] <- 0.999*V[[nm]] + 0.001*G[[nm]]^2
        P[[nm]] <- P[[nm]] - lr*(M[[nm]]/(1 - 0.9^t_))/(sqrt(V[[nm]]/(1 - 0.999^t_)) + 1e-8) } }
    hist[[ep]] <- c(ep, mean((ae_run(P, X[tr, ])$out - X[tr, ])^2), mean((ae_run(P, X[va, ])$out - X[va, ])^2)) }
  list(P = P, hist = do.call(rbind, hist)) }
Z128 <- svd_features(TRs, nc = 128)                      # the autoencoder works on the dense SVD space
t0 <- proc.time()[["elapsed"]]
ae <- ae_train(Z128, TRs, VAs, h = 64, k = 32, epochs = 40)
Zae <- ae_run(ae$P, Z128)$code
cat(sprintf("autoencoder (128 -> 64 -> 32 -> 64 -> 128) trained in %.1f s; reconstruction MSE on validation %.4f -> %.4f\n",
            proc.time()[["elapsed"]] - t0, ae$hist[1, 3], ae$hist[40, 3]))
aeh <- data.frame(epoch = ae$hist[, 1], training = ae$hist[, 2], validation = ae$hist[, 3])
g1 <- ggplot(aeh) + geom_line(aes(epoch, training, colour = "training")) +
  geom_line(aes(epoch, validation, colour = "validation")) + scale_y_log10() +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(y = "reconstruction MSE", title = "Unsupervised pretraining (surrogate)") +
  theme(legend.position = "bottom", legend.title = element_blank())
g2 <- ggplot(data.frame(d1 = Zae[, 1], d2 = Zae[, 2], pAct = y), aes(d1, d2, colour = pAct)) +
  geom_point(size = .5, alpha = .7) + scale_colour_viridis_c(name = "pActivity") +
  labs(x = "autoencoder dim 1", y = "autoencoder dim 2", title = "The embedding never saw the target") +
  theme(legend.position = "right", legend.key.height = unit(.4, "cm"))
print(g1 | g2)
cat(sprintf("Spearman correlation between autoencoder dimension 1 and pActivity: %+.3f\n",
            cor(Zae[, 1], y, method = "spearman")))



<p class="caption"><b>Figure 7.</b> Left: the surrogate autoencoder's reconstruction error on training and validation compounds; the two curves stay close, so the 32-number bottleneck is not memorising. Right: all compounds in the first two embedding dimensions, coloured by measured potency — the encoder never saw the target, and the printed rank correlation with potency is weak.</p>



In [ ]:
set.seed(0); sv32 <- irlba(FP[TRs, ], nv = 32); Zsvd32 <- as.matrix(FP %*% sv32$v)
reps <- list("raw 2048 bits" = Xfp, "SVD-32 embedding" = Zsvd32, "autoencoder-32 embedding" = Zae)
emb <- do.call(rbind, lapply(names(reps), function(nm) {
  F_ <- reps[[nm]]
  rr <- cv.glmnet(F_[TRs, ], y[TRs], alpha = 0, nfolds = 5)
  gb <- xgb.train(list(eta = 0.05, max_depth = 6, objective = "reg:squarederror", nthread = 4),
                  xgb.DMatrix(F_[TRs, ], label = y[TRs]), nrounds = 400, verbose = 0)
  data.frame(representation = nm, dimensions = ncol(F_),
             ridge_R2 = r2(y[TEs], as.numeric(predict(rr, newx = F_[TEs, ], s = "lambda.min"))),
             xgboost_R2 = r2(y[TEs], predict(gb, xgb.DMatrix(F_[TEs, ])))) }))
knitr::kable(emb, digits = 3, row.names = FALSE, caption = "Frozen-embedding workflow: three representations")
cat(sprintf("the two 32-dimensional codes lose %.3f (SVD) and %.3f (autoencoder) R2 against the raw bits with ridge\n",
            emb$ridge_R2[1] - emb$ridge_R2[2], emb$ridge_R2[1] - emb$ridge_R2[3]))


In [ ]:
dims <- c(16, 32, 64, 128, 256)
dimdf <- do.call(rbind, lapply(dims, function(d) {
  set.seed(0); svd_d <- irlba(FP[TRs, ], nv = d); Fd <- as.matrix(FP %*% svd_d$v)
  rr <- cv.glmnet(Fd[TRs, ], y[TRs], alpha = 0, nfolds = 3)
  gb <- xgb.train(list(eta = 0.05, max_depth = 6, objective = "reg:squarederror", nthread = 4),
                  xgb.DMatrix(Fd[TRs, ], label = y[TRs]), nrounds = 200, verbose = 0)
  data.frame(dimensions = d,
             ridge_R2 = r2(y[TEs], as.numeric(predict(rr, newx = Fd[TEs, ], s = "lambda.min"))),
             xgboost_R2 = r2(y[TEs], predict(gb, xgb.DMatrix(Fd[TEs, ])))) }))
g1 <- ggplot(emb, aes(representation)) +
  geom_col(aes(y = ridge_R2, fill = "ridge"), width = .38, position = position_nudge(x = -.2)) +
  geom_col(aes(y = xgboost_R2, fill = "xgboost"), width = .38, position = position_nudge(x = .2)) +
  scale_fill_manual(values = unname(PAL[c("blue", "green")])) + geom_hline(yintercept = 0, linewidth = .3) +
  labs(x = NULL, y = "test R^2 (scaffold split)", title = "Compression is not pretraining") +
  theme(axis.text.x = element_text(size = 6.5), legend.position = "bottom", legend.title = element_blank())
g2 <- ggplot(dimdf, aes(dimensions)) +
  geom_line(aes(y = ridge_R2, colour = "ridge on SVD")) + geom_point(aes(y = ridge_R2, colour = "ridge on SVD")) +
  geom_line(aes(y = xgboost_R2, colour = "xgboost on SVD")) +
  geom_point(aes(y = xgboost_R2, colour = "xgboost on SVD")) +
  geom_hline(aes(yintercept = emb$ridge_R2[1], colour = "ridge on raw bits"), linetype = 2) +
  geom_hline(aes(yintercept = emb$xgboost_R2[1], colour = "xgboost on raw bits"), linetype = 3) +
  scale_x_continuous(trans = "log2") + scale_colour_manual(values = unname(PAL)) +
  labs(y = "test R^2 (scaffold split)", title = "Accuracy against embedding width") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 5.6))
print(g1 | g2)
knitr::kable(dimdf, digits = 3, row.names = FALSE, caption = "Accuracy against SVD embedding width")



<p class="caption"><b>Figure 8.</b> Left: test R² of ridge and xgboost on the raw bits and on the two 32-dimensional codes. Right: the same two models on SVD embeddings of increasing width, with the raw-fingerprint results as horizontal references. Narrow codes lose accuracy for both models; ridge improves steadily with width and eventually passes its own raw-bit baseline (0.195 at 256 components against 0.105 on the raw bits — the projection also denoises), while xgboost, which handles sparse binary input well, never recovers its raw-bit score from any compressed code.</p>

The honest verdict of the surrogate experiment is a negative one, and it is exactly the right lesson to carry into reading the pretrained-model literature: **dimensionality reduction on your own small dataset is not transfer learning.** A 32-number code trained on two thousand molecules throws away information the downstream model could have used. What makes a real pretrained representation different is not the architecture of the encoder but the *corpus*: tens of millions of molecules or hundreds of millions of protein sequences, carrying statistics about chemistry that no single project's data contains<sup class="c"><a href="#r20">[20]</a><a href="#r21">[21]</a><a href="#r22">[22]</a></sup>. When you do have such embeddings, the workflow is the one coded above — freeze, embed once, model classically, and compare against the fingerprint baseline on the same split, because published comparisons find the advantage of learned representations over well-tuned fingerprint baselines to be modest and dataset-dependent<sup class="c"><a href="#r18">[18]</a><a href="#r19">[19]</a></sup>.

## 9. Graph neural networks and chemprop: pointers, not a tutorial

A **graph neural network** treats a molecule as a graph of atoms and bonds and learns a representation by passing messages along the bonds, so the featurisation is learned rather than hashed<sup class="c"><a href="#r17">[17]</a></sup>. `chemprop`, the directed message-passing implementation, is the standard reference tool and is what you should reach for if a project genuinely needs a learned molecular representation on tens of thousands of compounds; there is no R port, so this is a Python dependency in practice. Two findings temper the enthusiasm: on the MoleculeNet benchmarks graph networks are not uniformly better than descriptor or fingerprint models<sup class="c"><a href="#r19">[19]</a></sup>, and a careful comparison on bioactivity data found descriptor-based models competitive with graph models across most tasks<sup class="c"><a href="#r18">[18]</a></sup>.

## 10. A reading checklist for deep-learning papers

Apply this to every paper, including your own drafts. The six questions, with what a satisfactory answer looks like:

1. **What was the split?** Random, scaffold, time, or external? A random split on congeneric series answers a question nobody asked. Is the test set used only once?
2. **What is the baseline?** A fingerprint plus random forest or gradient boosting, tuned with the same budget as the neural model. A paper whose baseline is an untuned linear model has not made a comparison.
3. **How many runs?** A single seed is a single sample; architectures differ by less than seed-to-seed variation on small datasets.
4. **What was tuned, and on what?** Hyperparameters chosen on the test set invalidate the reported number.
5. **What compute, and what data?** GPU-hours and dataset size determine whether the result is reproducible by a reader and whether the comparison was fair.
6. **Is the code released, and are the weights?** Without them the result is an anecdote<sup class="c"><a href="#r29">[29]</a></sup>.

### 10.1 The checklist applied

Take two real papers, read together. Yang and co-workers' chemprop study<sup class="c"><a href="#r17">[17]</a></sup> reports a directed message-passing network against fingerprint and descriptor baselines on a large collection of public and proprietary datasets, with scaffold splits for several of them, released code, and an explicit discussion of hyperparameter optimisation — a strong answer to questions 1, 2, 4 and 6. Mayr and co-workers' large-scale comparison on ChEMBL<sup class="c"><a href="#r15">[15]</a></sup> concluded that feed-forward networks outperformed classical methods across thousands of assays. Robinson and co-workers then re-analysed that comparison<sup class="c"><a href="#r16">[16]</a></sup> and found that the conclusion depended on choices in the validation protocol; with corrected protocols the advantage of the neural models shrank considerably. The pair is the single best illustration in the field of why question 4 belongs on the list: the methods were not wrong, the comparison was.

<div class="exercise"><h4>Exercise 2</h4>
<p>Choose one deep-learning QSAR paper published in the last two years in a journal your group reads, and answer the six questions in writing, one short paragraph each, with page or figure references. Then reproduce its <em>baseline</em> (not its network) on our ROCK1 scaffold split with the code in Section 6, and state whether the paper's reported baseline performance is consistent with what a competent baseline achieves here. Bring the two paragraphs and the number to the next group meeting.</p></div>

<div class="box bridge"><h4>Bridge</h4>
<p>The chemistry-side treatment of neural models for molecules is the cheminformatics course's <a href="../../../Cheminformatics/U15_Deep_Learning/index.html">U15 (deep learning)</a>, with representations in <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">U5</a> and validation in <a href="../../../Cheminformatics/U14_Validation/index.html">U14</a>. Structure prediction — the clearest success of deep learning in this field — is the subject of <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">BGM324 unit 02</a>, and sequence analysis, the input of protein language models, of <a href="../../../BGM324/01_Sequence_Alignment/index.html">unit 01</a>. The artificial-intelligence course develops neural networks from the algorithmic side: <a href="../../../BGM411/R/index.html">BGM411 study material (R edition)</a>. Inside this course, <a href="../U15_Clustering/index.html">Unit 15</a> built the SVD and Tanimoto machinery that rescued the network here, <a href="../U16_Interpretability/index.html">Unit 16</a> explains how to interrogate whichever model wins, and <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> supplies the experimental floor that caps every number in this unit. The Python edition, with PyTorch, is <a href="../../Python/U18_Deep_Learning/index.html">here</a>.</p></div>

### 10.2 A run card for this unit

The same checklist applied to the work you have just read: the chunk below prints the configuration a reader would need to reproduce it.



In [ ]:
card <- sprintf('
RUN CARD - Unit 18 experiments
==============================
software     : R %s, nnet %s, xgboost %s, irlba %s, glmnet %s; CPU only
seeds        : set.seed(0) globally; every network initialised and shuffled from seed 1 unless swept;
               ranger is not used here, xgboost seeds set explicitly in section 6.1
data         : rock1_chembl_curated.csv, %d compounds with exact measurements; Morgan fingerprints
               radius 2 / 2048 bits read from the frozen sparse file (no RDKit in R)
splits       : scaffold (Bemis-Murcko, deterministic) %d/%d/%d; random %d/%d/%d; test used once per model
architecture : %d or 256 inputs -> 256 -> 64 -> 1, ReLU, dropout %.1f, AdamW lr %.0e,
               weight decay 1e-4, batch 64, early stopping patience 25, best weights restored
implementation: forward/backward written in base R matrices; gradient verified against central
               differences to %.0e; cross-checked against nnet on the same features
tuning       : %d MLP configurations and %d nnet configurations compared on the VALIDATION set only
runs         : 3 seeds for the headline comparison (section 6.1); spread reported
baselines    : xgboost (300 rounds, eta 0.05), ridge on raw bits, and the training mean
headline     : scaffold-split test R2 - MLP(SVD) %+.3f, xgboost %+.3f, MLP on raw bits %+.3f
pretrained   : NONE used; the embeddings in section 8 are an in-document surrogate (stated in the text)
compute      : whole unit a few minutes on a laptop CPU; no GPU anywhere
',
paste(R.version$major, R.version$minor, sep = "."), as.character(packageVersion("nnet")),
as.character(packageVersion("xgboost")), as.character(packageVersion("irlba")),
as.character(packageVersion("glmnet")), nrow(cmp), length(TRs), length(VAs), length(TEs),
length(TRr), length(VAr), length(TEr), ncol(Xfp), best_cfg$dropout, best_cfg$lr,
max(abs(ana - num)), nrow(sweep_tab), nrow(nnet_tab),
sc$test_R2[sc$model == "MLP on 256 scaled SVD components"],
sc$test_R2[sc$model == "xgboost on raw 2048 bits"],
sc$test_R2[sc$model == "MLP on raw 2048 bits"])
cat(card)


In [ ]:
summary_tab <- data.frame(
  experiment = c("backpropagation by hand vs finite differences", "MLP on raw bits (scaffold split)",
                 "regularisation (none)", "MLP on 256 scaled SVD components",
                 "nnet on the same features", "xgboost on raw bits (scaffold split)",
                 "random vs scaffold split (xgboost)", "learning curves",
                 "surrogate embeddings (32 dims)"),
  result = c(sprintf("agreement %.0e", max(abs(ana - num))),
             sprintf("R2 %+.3f", sc$test_R2[sc$model == "MLP on raw 2048 bits"]),
             sprintf("train MSE %.3f vs val %.3f", reg$final_train_MSE[1], reg$final_val_MSE[1]),
             sprintf("R2 %+.3f", sc$test_R2[sc$model == "MLP on 256 scaled SVD components"]),
             sprintf("best test R2 %+.3f", max(nnet_tab$test_R2)),
             sprintf("R2 %+.3f", sc$test_R2[sc$model == "xgboost on raw 2048 bits"]),
             sprintf("%+.3f vs %+.3f",
                     comp$test_R2[comp$split == "random" & comp$model == "xgboost on raw 2048 bits"],
                     sc$test_R2[sc$model == "xgboost on raw 2048 bits"]),
             sprintf("xgboost %+.2f -> %+.2f over %d-%d compounds", lc$xgboost[1], lc$xgboost[4],
                     sizes[1], sizes[4]),
             sprintf("ridge %+.3f (raw) vs %+.3f (autoencoder)", emb$ridge_R2[1], emb$ridge_R2[3])),
  reading = c("the chain rule, verified once", "fails: sparse binary input, ~2,000 compounds",
              "memorises the training set", "representation fixed what dropout could not",
              "an independent implementation agrees", "still ahead, with no tuning",
              "protocol matters more than architecture", "data is the binding constraint",
              "compression is not pretraining"))
knitr::kable(summary_tab, row.names = FALSE)



<p class="caption"><b>Table 1.</b> The nine experiments of this unit. Several are negative results about neural networks on small molecular datasets; all of them are reported, because they are the ones that change what a student should do next.</p>

## 11. Summary

**The mechanics are simple and worth knowing exactly.** An MLP is affine maps composed with ReLU; backpropagation is the chain rule in reverse, verified here against central differences to eleven decimal places; Adam with decoupled weight decay and mini-batches is the optimiser you will use<sup class="c"><a href="#r5">[5]</a><a href="#r6">[6]</a></sup>; a correct training loop evaluates with dropout off, stops on a validation set and restores the best weights<sup class="c"><a href="#r7">[7]</a></sup>. Written out in base R matrices it is about sixty lines, and `nnet`<sup class="c"><a href="#r31">[31]</a></sup> reaches the same place on the same features.

**Regularisation changes the gap, not always the answer.** Without dropout or weight decay the network drove its training loss towards zero while its validation loss deteriorated — textbook overfitting. With them, a third of the gap closed and the held-out score rose from −0.40 to about −0.03, still below the training-mean baseline: the model was failing to transfer across scaffolds, which no penalty repairs.

**Representation was decisive.** Projecting 2,048 sparse bits onto 256 standardised SVD components moved the scaffold-split R² from −0.21 to +0.09 with the architecture unchanged — positive at last, and still well behind gradient boosting. On sparse binary molecular inputs, preprocessing is the hyperparameter that matters.

**Gradient boosting still wins on data of this size and shape**, on both splits, with no tuning — the tabular-data finding<sup class="c"><a href="#r13">[13]</a><a href="#r14">[14]</a></sup> and the corrected cheminformatics comparison<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup>, reproduced on the group's own target. The learning curves have not flattened, so the remedy is more data, not more layers.

**Pretrained representations are the part of deep learning most worth borrowing**, and the workflow is freeze-embed-model. The offline surrogate used here — an SVD code and an autoencoder trained on two thousand fingerprints — is explicitly *not* a pretrained model, and its loss of accuracy relative to the raw bits makes the point that the value of ChemBERTa-type and ESM-2-type representations comes from the size and diversity of the pretraining corpus<sup class="c"><a href="#r20">[20]</a><a href="#r22">[22]</a></sup>, not from compression.

**Reading is a skill with a checklist**: split, baseline, number of runs, what was tuned and on what, compute and data, released code<sup class="c"><a href="#r29">[29]</a></sup>. The Mayr-versus-Robinson pair<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup> is the case study to remember when a result looks decisive. Everything here ran in base R<sup class="c"><a href="#r33">[33]</a></sup> with `xgboost`<sup class="c"><a href="#r32">[32]</a></sup>, `glmnet`<sup class="c"><a href="#r34">[34]</a></sup>, `irlba` and `ggplot2`<sup class="c"><a href="#r35">[35]</a></sup>, on a laptop CPU.

## 12. Self-assessment

<details><summary>1. Why does an MLP without a non-linear activation have no more expressive power than a linear model?</summary>
<p>Composing affine maps gives another affine map: W2(W1x + b1) + b2 = (W2W1)x + (W2b1 + b2). Depth without a non-linearity is a reparameterisation, not a richer hypothesis class. The activation is what breaks the composition.</p></details>

<details><summary>2. In the hand-worked example, why does the gradient pass through the ReLU units unchanged?</summary>
<p>Both pre-activations were positive (0.75 and 0.55), and the derivative of ReLU is 1 for positive inputs, so the elementwise multiplication by phi'(z) leaves delta unchanged. Had a unit been negative, its gradient would have been exactly zero — the "dying ReLU" that a too-large learning rate produces.</p></details>

<details><summary>3. You wrote your own backward pass. How do you know it is right?</summary>
<p>By a finite-difference check: perturb each parameter by ±1e-6, recompute the loss, and compare (L(θ+ε) − L(θ−ε))/2ε with the analytic gradient. In Section 3 the largest discrepancy over nine parameters was of order 1e-11. Any custom layer should be checked this way before it is trained; a framework's autograd is tested the same way.</p></details>

<details><summary>4. Distinguish what dropout, weight decay and early stopping each do.</summary>
<p>Dropout removes units at random during training so no unit can depend on another, acting like an ensemble of sub-networks. Weight decay shrinks weights towards zero, preferring smoother functions. Early stopping limits the effective capacity by halting before the model exploits the noise. In Section 5 all three reduced the train–validation gap; none of them produced a competitive scaffold-split score.</p></details>

<details><summary>5. The MLP on raw fingerprint bits had a negative R² while the same architecture on 256 scaled SVD components was positive. Explain.</summary>
<p>Each of the 2,048 input coordinates is a sparse, unordered indicator, so the first layer must estimate 2,048 weights per unit from fewer than 2,000 examples with no structure to share statistical strength. The SVD projection produces a few hundred dense, decorrelated and comparably scaled inputs, which conditions the optimisation (gradients of similar magnitude for every input) and reduces the number of effective parameters. The lesson is that preprocessing is part of the architecture.</p></details>

<details><summary>6. Why must the SVD be fitted on the training set only?</summary>
<p>Because fitting it on all compounds lets information from the test set into the feature construction — leakage of the same kind as scaling before splitting. The projection is part of the model and must be refitted inside every fold or split, which is why <code>svd_features()</code> takes the training indices as an argument.</p></details>

<details><summary>7. What do learning curves that have not flattened tell a project manager?</summary>
<p>That the dataset is the binding constraint: more measurements of the same kind would still improve the model, and switching model class would not. In Section 7 all three model classes kept improving up to the full training set, with a stable ordering between them.</p></details>

<details><summary>8. Describe the frozen-embedding workflow in three steps, and say when it helps.</summary>
<p>Pretrain (or download) a large self-supervised model; run one forward pass per molecule or sequence to extract embeddings with the weights frozen; fit a classical model on those embeddings. It helps most when the labelled set is small and the pretraining corpus covers the relevant chemistry or protein family, and least on large homogeneous single-target datasets.</p></details>

<details><summary>9. Why is the autoencoder in Section 8 not a pretrained model?</summary>
<p>Because it was trained on about two thousand molecules from this very dataset, so it contains no information beyond what the downstream model already has — and compresses it. A real pretrained representation derives its value from a corpus of tens of millions of molecules or hundreds of millions of sequences. The unit states this substitution explicitly because no pretrained weights can be downloaded in the course's offline environment.</p></details>

<details><summary>10. List the six questions of the reading checklist from memory, in any order.</summary>
<p>Split (and whether the test set was used once); baseline (and whether it was tuned with the same budget); number of runs or seeds; what was tuned and on which data; compute and dataset size; availability of code and weights.</p></details>

<details><summary>11. A paper reports that its graph neural network beats a random-forest baseline by 0.03 in ROC-AUC on a random split, single seed. What do you ask for?</summary>
<p>Seed-to-seed variability for both models — Section 6.1 measured it here and the spread was of the same order as many published differences — a scaffold or time split, confirmation that the baseline's hyperparameters were tuned with comparable effort, the size of the dataset and the compute used, and the code. Without those the result is not yet a comparison.</p></details>

<details><summary>12. When would you actually choose a neural network for a project in this laboratory?</summary>
<p>When a pretrained representation is available and the labelled set is small; when the data is large (tens of thousands of compounds or more) and a learned representation can be trained properly; when the task is multi-task or multi-modal (several assays, sequence plus structure); or when the output is not a scalar — generative design, structure prediction. Not for a few thousand rows of single-target bioactivity with hashed fingerprints, where Section 6 shows gradient boosting ahead at a fraction of the cost.</p></details>

## Glossary

<dl class="gloss">
<dt>Activation function</dt><dd>Elementwise non-linearity (ReLU, tanh, sigmoid) applied after an affine layer; without it, depth collapses to a single linear map.</dd>
<dt>Adam / AdamW</dt><dd>Adaptive optimiser using running first and second moments of the gradient; AdamW decouples the weight-decay term from the moment estimates.</dd>
<dt>Autoencoder</dt><dd>Network trained to reconstruct its own input through a bottleneck; its bottleneck activations are an unsupervised embedding.</dd>
<dt>Backpropagation</dt><dd>Reverse-mode application of the chain rule that computes all parameter gradients at the cost of about one forward pass.</dd>
<dt>Dropout</dt><dd>Randomly zeroing hidden units during training only, rescaling the survivors, to prevent co-adaptation.</dd>
<dt>Early stopping</dt><dd>Halting training when the validation loss stops improving, and restoring the weights of the best epoch.</dd>
<dt>Embedding (frozen)</dt><dd>Internal activations of a pretrained model used as fixed features for a downstream classical model.</dd>
<dt>Epoch / mini-batch</dt><dd>One pass over the training set / the subset of examples used for one gradient step.</dd>
<dt>Finite-difference gradient check</dt><dd>Comparing an analytic gradient with (L(θ+ε) − L(θ−ε))/2ε; the standard test for a hand-written backward pass.</dd>
<dt>Fine-tuning</dt><dd>Continuing to train a pretrained model's weights on the downstream task, as opposed to freezing them.</dd>
<dt>Graph neural network</dt><dd>Network that operates on the molecular graph by passing messages along bonds, learning the featurisation instead of hashing it.</dd>
<dt>He initialisation</dt><dd>Drawing weights with variance 2/n_in so that activations neither vanish nor explode across ReLU layers.</dd>
<dt>Learning curve</dt><dd>Held-out performance as a function of training-set size, with validation and test sets held fixed.</dd>
<dt>Learning rate</dt><dd>Step size of the optimiser; the hyperparameter with the largest effect on whether training works at all.</dd>
<dt>Multilayer perceptron (MLP)</dt><dd>Fully connected feed-forward network of affine layers and activations.</dd>
<dt>Pretraining (self-supervised)</dt><dd>Training on a label-free objective over a large corpus, to be reused downstream.</dd>
<dt>Weight decay</dt><dd>L2 penalty on the parameters, added to the gradient at each step.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">LeCun Y, Bengio Y, Hinton G. Deep learning. <em>Nature</em>. 2015;521(7553):436-444. <a href="https://doi.org/10.1038/nature14539">doi:10.1038/nature14539</a></li>
<li id="r2">Rumelhart DE, Hinton GE, Williams RJ. Learning representations by back-propagating errors. <em>Nature</em>. 1986;323(6088):533-536. <a href="https://doi.org/10.1038/323533a0">doi:10.1038/323533a0</a></li>
<li id="r3">Robbins H, Monro S. A stochastic approximation method. <em>Ann Math Stat</em>. 1951;22(3):400-407. <a href="https://doi.org/10.1214/aoms/1177729586">doi:10.1214/aoms/1177729586</a></li>
<li id="r4">Bottou L. Large-scale machine learning with stochastic gradient descent. In: <em>Proc. COMPSTAT 2010</em>. Physica-Verlag; 2010:177-186. <a href="https://doi.org/10.1007/978-3-7908-2604-3_16">doi:10.1007/978-3-7908-2604-3_16</a></li>
<li id="r5">Kingma DP, Ba J. Adam: a method for stochastic optimization. <em>arXiv</em>. 2014:1412.6980. <a href="https://doi.org/10.48550/arXiv.1412.6980">doi:10.48550/arXiv.1412.6980</a></li>
<li id="r6">Loshchilov I, Hutter F. Decoupled weight decay regularization. <em>arXiv</em>. 2017:1711.05101. <a href="https://doi.org/10.48550/arXiv.1711.05101">doi:10.48550/arXiv.1711.05101</a></li>
<li id="r7">Prechelt L. Early stopping - but when? In: <em>Neural Networks: Tricks of the Trade</em>. LNCS 1524. Springer; 1998:55-69. <a href="https://doi.org/10.1007/3-540-49430-8_3">doi:10.1007/3-540-49430-8_3</a></li>
<li id="r8">He K, Zhang X, Ren S, Sun J. Delving deep into rectifiers: surpassing human-level performance on ImageNet classification. In: <em>Proc. IEEE ICCV</em>; 2015:1026-1034. <a href="https://doi.org/10.1109/ICCV.2015.123">doi:10.1109/ICCV.2015.123</a></li>
<li id="r9">Hinton GE, Srivastava N, Krizhevsky A, Sutskever I, Salakhutdinov RR. Improving neural networks by preventing co-adaptation of feature detectors. <em>arXiv</em>. 2012:1207.0580. <a href="https://doi.org/10.48550/arXiv.1207.0580">doi:10.48550/arXiv.1207.0580</a></li>
<li id="r10">Zhang C, Bengio S, Hardt M, Recht B, Vinyals O. Understanding deep learning (still) requires rethinking generalization. <em>Commun ACM</em>. 2021;64(3):107-115. <a href="https://doi.org/10.1145/3446776">doi:10.1145/3446776</a></li>
<li id="r11">Ioffe S, Szegedy C. Batch normalization: accelerating deep network training by reducing internal covariate shift. <em>arXiv</em>. 2015:1502.03167. <a href="https://doi.org/10.48550/arXiv.1502.03167">doi:10.48550/arXiv.1502.03167</a></li>
<li id="r12">Baydin AG, Pearlmutter BA, Radul AA, Siskind JM. Automatic differentiation in machine learning: a survey. <em>J Mach Learn Res</em>. 2018;18(153):1-43. <a href="https://doi.org/10.48550/arXiv.1502.05767">doi:10.48550/arXiv.1502.05767</a></li>
<li id="r13">Grinsztajn L, Oyallon E, Varoquaux G. Why do tree-based models still outperform deep learning on tabular data? <em>arXiv</em>. 2022:2207.08815. <a href="https://doi.org/10.48550/arXiv.2207.08815">doi:10.48550/arXiv.2207.08815</a></li>
<li id="r14">Shwartz-Ziv R, Armon A. Tabular data: deep learning is not all you need. <em>Inf Fusion</em>. 2022;81:84-90. <a href="https://doi.org/10.1016/j.inffus.2021.11.011">doi:10.1016/j.inffus.2021.11.011</a></li>
<li id="r15">Mayr A, Klambauer G, Unterthiner T, et al. Large-scale comparison of machine learning methods for drug target prediction on ChEMBL. <em>Chem Sci</em>. 2018;9(24):5441-5451. <a href="https://doi.org/10.1039/C8SC00148K">doi:10.1039/C8SC00148K</a></li>
<li id="r16">Robinson MC, Glen RC, Lee AA. Validating the validation: reanalyzing a large-scale comparison of deep learning and machine learning models for bioactivity prediction. <em>J Comput Aided Mol Des</em>. 2020;34:717-730. <a href="https://doi.org/10.1007/s10822-019-00274-0">doi:10.1007/s10822-019-00274-0</a></li>
<li id="r17">Yang K, Swanson K, Jin W, et al. Analyzing learned molecular representations for property prediction. <em>J Chem Inf Model</em>. 2019;59(8):3370-3388. <a href="https://doi.org/10.1021/acs.jcim.9b00237">doi:10.1021/acs.jcim.9b00237</a></li>
<li id="r18">Jiang D, Wu Z, Hsieh CY, et al. Could graph neural networks learn better molecular representation for drug discovery? A comparison study of descriptor-based and graph-based models. <em>J Cheminform</em>. 2021;13:12. <a href="https://doi.org/10.1186/s13321-020-00479-8">doi:10.1186/s13321-020-00479-8</a></li>
<li id="r19">Wu Z, Ramsundar B, Feinberg EN, et al. MoleculeNet: a benchmark for molecular machine learning. <em>Chem Sci</em>. 2018;9(2):513-530. <a href="https://doi.org/10.1039/C7SC02664A">doi:10.1039/C7SC02664A</a></li>
<li id="r20">Chithrananda S, Grand G, Ramsundar B. ChemBERTa: large-scale self-supervised pretraining for molecular property prediction. <em>arXiv</em>. 2020:2010.09885. <a href="https://doi.org/10.48550/arXiv.2010.09885">doi:10.48550/arXiv.2010.09885</a></li>
<li id="r21">Ross J, Belgodere B, Chenthamarakshan V, Padhi I, Mroueh Y, Das P. Large-scale chemical language representations capture molecular structure and properties. <em>Nat Mach Intell</em>. 2022;4(12):1256-1264. <a href="https://doi.org/10.1038/s42256-022-00580-7">doi:10.1038/s42256-022-00580-7</a></li>
<li id="r22">Lin Z, Akin H, Rao R, et al. Evolutionary-scale prediction of atomic-level protein structure with a language model. <em>Science</em>. 2023;379(6637):1123-1130. <a href="https://doi.org/10.1126/science.ade2574">doi:10.1126/science.ade2574</a></li>
<li id="r23">Rives A, Meier J, Sercu T, et al. Biological structure and function emerge from scaling unsupervised learning to 250 million protein sequences. <em>Proc Natl Acad Sci USA</em>. 2021;118(15):e2016239118. <a href="https://doi.org/10.1073/pnas.2016239118">doi:10.1073/pnas.2016239118</a></li>
<li id="r24">Jumper J, Evans R, Pritzel A, et al. Highly accurate protein structure prediction with AlphaFold. <em>Nature</em>. 2021;596:583-589. <a href="https://doi.org/10.1038/s41586-021-03819-2">doi:10.1038/s41586-021-03819-2</a></li>
<li id="r25">Abramson J, Adler J, Dunger J, et al. Accurate structure prediction of biomolecular interactions with AlphaFold 3. <em>Nature</em>. 2024;630:493-500. <a href="https://doi.org/10.1038/s41586-024-07487-w">doi:10.1038/s41586-024-07487-w</a></li>
<li id="r26">Hinton GE, Salakhutdinov RR. Reducing the dimensionality of data with neural networks. <em>Science</em>. 2006;313(5786):504-507. <a href="https://doi.org/10.1126/science.1127647">doi:10.1126/science.1127647</a></li>
<li id="r27">Walters WP, Barzilay R. Applications of deep learning in molecule generation and molecular property prediction. <em>Acc Chem Res</em>. 2021;54(2):263-270. <a href="https://doi.org/10.1021/acs.accounts.0c00699">doi:10.1021/acs.accounts.0c00699</a></li>
<li id="r28">Bender A, Cortes-Ciriano I. Artificial intelligence in drug discovery: what is realistic, what are illusions? Part 1. <em>Drug Discov Today</em>. 2021;26(2):511-524. <a href="https://doi.org/10.1016/j.drudis.2020.12.009">doi:10.1016/j.drudis.2020.12.009</a></li>
<li id="r29">Haibe-Kains B, Adam GA, Hosny A, et al. Transparency and reproducibility in artificial intelligence. <em>Nature</em>. 2020;586:E14-E16. <a href="https://doi.org/10.1038/s41586-020-2766-y">doi:10.1038/s41586-020-2766-y</a></li>
<li id="r30">van Tilborg D, Alenicheva A, Grisoni F. Exposing the limitations of molecular machine learning with activity cliffs. <em>J Chem Inf Model</em>. 2022;62(23):5938-5951. <a href="https://doi.org/10.1021/acs.jcim.2c01073">doi:10.1021/acs.jcim.2c01073</a></li>
<li id="r31">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r32">Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: <em>Proc. 22nd ACM SIGKDD</em>; 2016:785-794. <a href="https://doi.org/10.1145/2939672.2939785">doi:10.1145/2939672.2939785</a></li>
<li id="r33">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r34">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r35">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Goodfellow I, Bengio Y, Courville A. <em>Deep Learning</em>. MIT Press; 2016 (free online). - Chapters 6-8 are the reference for the mathematics in Sections 2-5.</li>
<li>Prince SJD. <em>Understanding Deep Learning</em>. MIT Press; 2023 (free PDF). - A modern, well-illustrated replacement with exercises.</li>
<li>White AD. <em>Deep Learning for Molecules and Materials</em>. 2021 (free online, dmol.pub). - The chemistry-facing companion, including graph networks.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em><sup class="c"><a href="#r31">[31]</a></sup>. - Chapter 8 documents <code>nnet</code>, the network used as a cross-check here. Batch normalisation<sup class="c"><a href="#r11">[11]</a></sup>, not used in this unit, is covered in the deep-learning texts above.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>chemprop documentation and repository (Python): <a href="https://chemprop.readthedocs.io/">chemprop.readthedocs.io</a>.</li>
<li>Hugging Face model cards for ChemBERTa and ESM-2 (the weights this environment could not download): <a href="https://huggingface.co/models">huggingface.co/models</a>.</li>
<li>CRAN Task View "Machine Learning &amp; Statistical Learning", neural-network section: <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts comparing deep models with fingerprint baselines: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
